# SQL - CASE STATEMENT

#### Evaluates a list of conditions and returns a value when the first condition is met

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_customers = pd.read_csv('data/sales_customers.csv')
df_employees = pd.read_csv('data/sales_employees.csv')
df_orders = pd.read_csv('data/sales_orders.csv')
df_orderarchive = pd.read_csv('data/sales_ordersarchive.csv')
df_products = pd.read_csv('data/sales_products.csv')

### Main purpose is Data Transoformartion Derive new information

- Create new Columns based on existing data

### SQL TASK

#### Create a report showing total sales for each of the following categories: High (sales over 50), Medion (sales 21 - 50), And LOW (Sales 20 or less)
#### Sort the categories from highest sales to lowest

```SQL
SELECT
    Category,
    SUM(Sales) AS TotalSales
FROM (
    SELECT
        orderid,
        sales,
        CASE
            WHEN sales > 50 THEN 'High'
            WHEN sales > 20 THEN 'Medium'
            ELSE 'Low'
        END AS Category
    FROM sales.orders) t
GROUP BY Category
ORDER BY TotalSales DESC
```

In [8]:
df_or = df_orders.copy()

bins = [-float('inf'),20,50,float('inf')]
labels = ['Low','Medium','High']

df_or['Category'] = pd.cut(df_or['sales'], bins=bins, labels=labels)

result = (
    df_or.groupby('Category', observed=False)['sales']
    .sum()
    .reset_index(name='TotalSales')
    .sort_values(by='TotalSales',ascending=False)
)

result

,Category,TotalSales
2,High,210
1,Medium,105
0,Low,65


### MAPPING

#### Transform the values from one form to another

### SQL TASK

#### Retrieve employee details with gender displayed as full text

```SQL
SELECT
    firstname,
    lastname,
    gender,
    CASE
        WHEN gender = 'F' THEN 'Female'
        WHEN gender = 'M' THEN 'Male'
        ELSE 'Not Avaliable'
    END AS GenderFullText
FROM sales.employees
```
### SQL TASK

#### Retrieve customer details with abbreviated country code

```SQL
SELECT
    country,
    CASE
        WHEN country = 'USA'THEN 'US'
        WHEN country = 'Germany' THEN 'DE'
        WHEN country = 'ITALY' THEN 'IT'
        WHEN country = 'FRANCE' THEN 'FR'
    END AS countryAbreviation
FROM sales.customers;

SELECT
    country,
    CASE country
        WHEN 'USA' THEN 'US'
        WHEN 'Germany' THEN 'DE'
        ELSE 'n/a'
    END as countryAbreviation
FROM sales.customers;
```

### HANDLING NULLS

#### Replace NULLs with a specific value

- NULLS cal lead to inaccurate results which cal lead to wrong decision-making

### SQL TASK

#### Find the average scores of customers and treeat Nulls as 0

```SQL
SELECT
    customerid,
    lastname,
    score,
    AVG(CASE WHEN SCORE IS NULL THEN 0 ELSE score END) OVER() AvgCustomer
FROM sales.customers
```

In [19]:
df = df_customers.copy()

score_filled = df['score'].fillna(0)

df['AvgCustomer'] = score_filled.mean()

result = df[['customerid','lastname','score','AvgCustomer']]

result

,customerid,lastname,score,AvgCustomer
0,1,Goldberg,350.0,500.0
1,2,Brown,900.0,500.0
2,3,NaN,750.0,500.0
3,4,Schwarz,500.0,500.0
4,5,Adams,NaN,500.0


### CONDITIONAL AGGREGATION

#### Apply aggregate functions only on subsets of data that fulfull certain conditions

### SQL TASK

#### Count how many times each customer has made an order with sales greater than 30 and total sales for each customer

```SQL
SELECT
    customerid,
    SUM(CASE WHEN sales > 30 THEN 1 ELSE 0 END) AS TotalOrdersHigh,
    COUNT(*) TotalOrdes
FROM sales.orders
GROUP BY customerid;
```

In [20]:
df_or = df_orders.copy()

df_or['is_high'] = df_or['sales'] > 30

result = df_or.groupby('customerid').agg(
    TotalOrdersHigh=('is_high', 'sum'),
    TotalOrders=('sales','size')
).reset_index()

result

,customerid,TotalOrdersHigh,TotalOrders
0,1,1,3
1,2,0,3
2,3,2,3
3,4,1,1
